# Week 6 · Task D — Bonus (D1: make it interactive)
**5 marks, reported separately.** Chosen option: **D1 — rebuild one Task B chart in Plotly** so the reader can hover a value, filter a category and zoom a date range.

I rebuild **Chart 1 (monthly energy delivered by zone)** because it is the chart with a real time axis and six categories worth filtering. The interaction adds three things the static panels cannot: **hover** (exact MWh/day, total MWh, days covered, and whether the month is partial or estimated), **filter** (click a zone in the legend, or use the dropdown, to isolate it) and **zoom** (drag the range slider or box-zoom on any months).

> Run **Task A** first. Output: `partD_interactive.html` (opens in any browser; loads Plotly from a CDN, so it needs an internet connection).

In [2]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

def find_root():
    """Folder that holds the three toolkits and data/ - works from week6/assignments or a copy of it."""
    here = Path.cwd().resolve()
    for base in [here, here / "assignments", here.parent, here.parent / "assignments"]:
        if (base / "vizlib.py").exists() and (base / "data").is_dir():
            return base
    raise FileNotFoundError("Open this notebook in the folder that contains vizlib.py, tslib.py, geolib.py and data/")

ROOT = find_root()
sys.path.insert(0, str(ROOT))
DATA, CLEAN = ROOT / "data", ROOT / "clean"
CLEAN.mkdir(exist_ok=True)

import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from vizlib import PALETTE
pio.renderers.default = "notebook_connected"
assert (CLEAN / "grid_monthly_energy.csv").exists(), "Run Task_A_process.ipynb first"
mm = pd.read_csv(CLEAN / "grid_monthly_energy.csv")
mm["m"] = pd.to_datetime(mm.month + "-01")
zones = sorted(mm.zone.unique())
print(len(mm), "zone-months |", mm.partial.sum(), "partial |", mm.estimated.sum(), "estimated |", zones)

108 zone-months | 12 partial | 1 estimated | ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']


In [3]:
# Okabe-Ito colours (colour-blind safe), one per zone; zone name is ALSO written in the legend and on hover, so colour is never the only cue
zone_col = dict(zip(zones, ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]))
fig = go.Figure()
trace_zone = []                                   # which zone each trace belongs to, for the dropdown
for z in zones:
    g = mm[mm.zone == z].sort_values("m"); c = zone_col[z]
    cd = np.column_stack([g.energy_mwh, g.days_covered, g.partial, g.estimated, g.n_gap_days])
    ht = ("<b>Zone " + z + "</b> \u00b7 %{x|%b %Y}<br>%{y:,.0f} MWh per day<br>month total %{customdata[0]:,.0f} MWh over %{customdata[1]} days"
          "<br>unreported days: %{customdata[4]}<extra></extra>")
    fig.add_trace(go.Scatter(x=g.m, y=g.mean_daily_mwh, mode="lines+markers", name=f"Zone {z}", legendgroup=z, line=dict(color=c, width=2.4),
                             marker=dict(size=5, color=c), customdata=cd, hovertemplate=ht)); trace_zone.append(z)
    p = g[g.partial]
    fig.add_trace(go.Scatter(x=p.m, y=p.mean_daily_mwh, mode="markers", legendgroup=z, showlegend=False, marker=dict(size=10, color="white", line=dict(color=c, width=2)),
                             customdata=np.column_stack([p.energy_mwh, p.days_covered, p.partial, p.estimated, p.n_gap_days]),
                             hovertemplate="<b>Zone " + z + "</b> \u00b7 %{x|%b %Y} <i>(partial month, %{customdata[1]} days)</i><br>%{y:,.0f} MWh per day<extra></extra>")); trace_zone.append(z)
    e = g[g.estimated]
    if len(e):
        fig.add_trace(go.Scatter(x=e.m, y=e.mean_daily_mwh, mode="markers", legendgroup=z, showlegend=False,
                                 marker=dict(size=11, symbol="diamond", color="white", line=dict(color="#D55E00", width=2.2)),
                                 hovertemplate="<b>Zone " + z + "</b> \u00b7 %{x|%b %Y} <i>(one meter-reset day is estimated)</i><br>%{y:,.0f} MWh per day<extra></extra>")); trace_zone.append(z)

# dropdown filter: All, or one zone
buttons = [dict(label="All zones", method="update", args=[{"visible": [True] * len(trace_zone)}])]
for z in zones:
    buttons.append(dict(label=f"Only {z}", method="update", args=[{"visible": [tz == z for tz in trace_zone]}]))

full = mm[~mm.partial]
peak_name = pd.Timestamp(full.loc[full.mean_daily_mwh.idxmax(), "month"] + "-01").strftime("%B")
trough_name = pd.Timestamp(full.loc[full.mean_daily_mwh.idxmin(), "month"] + "-01").strftime("%B")
fig.update_layout(
    title=dict(text=f"Every zone's daily energy peaks in {peak_name} and bottoms out in {trough_name} \u2014 hover, filter or zoom to read any month",
               x=0.0, xanchor="left", font=dict(size=16)),
    template="simple_white", height=620, margin=dict(l=70, r=30, t=90, b=150), hovermode="closest",
    yaxis=dict(title="Energy delivered (MWh per day)", gridcolor="#DFE6EE", rangemode="tozero"),
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.07), gridcolor="#EEF2F6", title=None),
    legend=dict(title="click a zone to hide / show it<br>double-click to isolate", orientation="v"),
    updatemenus=[dict(buttons=buttons, direction="down", x=0.0, xanchor="left", y=1.12, yanchor="top", showactive=True)],
    annotations=[dict(xref="paper", yref="paper", x=0, y=-0.43, xanchor="left", yanchor="top", showarrow=False, align="left", font=dict(size=10, color="#5B6573"),
        text=("Declared: monthly energy = change in the meter reading (month-end minus previous month-end), shown as mean MWh/day over the days covered. "
              "Hollow circles = partial months (Apr 2025: 29 days, Sep 2026: 11 days); diamond = P4 Feb 2026, one meter-reset day estimated.<br>"
              "128 never-reported zone-days are left as gaps, not zeros. One y-axis for all zones, starting at zero (rangemode tozero), no second axis. "
              "Source: National Grid Authority daily returns (cleaned by Task A), synthetic data CDB601220."))])
fig.write_html(ROOT / "partD_interactive.html", include_plotlyjs="cdn", full_html=True)
print("wrote", (ROOT / "partD_interactive.html").name, f"({(ROOT / 'partD_interactive.html').stat().st_size/1024:.0f} KB)")
fig

wrote partD_interactive.html (29 KB)


In [4]:
# a static copy of the interactive view, if the image exporter is available on this machine (optional)
try:
    fig.write_image(str(ROOT / "partD_interactive_static.png"), width=1300, height=650, scale=2)
    print("wrote partD_interactive_static.png")
except Exception as ex:
    print("static export skipped (needs the kaleido + Chrome image exporter):", type(ex).__name__)

static export skipped (needs the kaleido + Chrome image exporter): RuntimeError


## The two sentences the brief asks for

**What the interaction lets a reader do that the static version cannot:** pick out any single zone-month and read its exact MWh/day, total MWh and days covered, isolate one zone with the legend or dropdown, and zoom the range slider onto a few months — none of which fits legibly on six small static panels.

**What it lets them do that I would rather they did not:** zoom to a three-month window or isolate a small zone and read the ordinary seasonal swing as a dramatic trend (or P5's wiggle as large as P6's), because the y-axis rescales to whatever is left — which is why the y-axis is pinned to start at zero, and why the static version, with its fixed windows, remains the version for the board pack.

In [5]:
(ROOT / "partD_justification.md").write_text(
"""# Part D (D1) - justification

**Chosen:** D1, Task B chart 1 (monthly energy delivered by zone) rebuilt in Plotly -> partD_interactive.html

**What the interaction lets a reader do that the static version cannot:** pick out any single zone-month and read its exact MWh/day, total MWh and days covered, isolate one zone with the legend or dropdown, and zoom the range slider onto a few months - none of which fits legibly on six small static panels.

**What it lets them do that I would rather they did not:** zoom to a three-month window or isolate a small zone and read the ordinary seasonal swing as a dramatic trend (or P5's wiggle as large as P6's), because the y-axis rescales to whatever is left - which is why the y-axis is pinned to start at zero, and why the static version, with its fixed windows, remains the version for the board pack.
""", encoding="utf-8")
print("wrote partD_justification.md")

wrote partD_justification.md
